In [ ]:
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import geoopt
from sklearn.metrics import confusion_matrix, f1_score
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from sklearn.decomposition import PCA

class Config:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    batch_size = 128
    learning_rate = 2e-4
    epochs = 100
    
    input_dim = 1280       # Placeholder
    hyp_dim = 64          # Dimensione dello spazio iperbolico
    
    classifier_layers = [512, 256, 128] 
    
    output_dim = 5     

    c = 1.0           #parametro di curvatura dello spazio iperbolico  
    cone_aperture_k = 1.5
    lambda_entail = 6
    
    interval = [-np.inf, -2, -0.5, 0.5, 2, np.inf] 
    inter_names = ['Destroyed', 'Deleterious', 'Neutral', 'Beneficial', 'Improved']
    label_col = ['#d73027', '#fc8d59', '#fee090', '#91bfdb', '#4575b4']

def discretize_fitness(fitness_array):
    bins = Config.interval
    labels = np.digitize(fitness_array, bins) - 1
    return np.clip(labels, 0, len(Config.inter_names) - 1)

def compute_class_weights(loader, device):
    counts = torch.zeros(len(Config.inter_names)).to(device)
    for _, labels, _ in loader:
        labels = labels.to(device)
        counts += torch.bincount(labels, minlength=len(Config.inter_names))
    weights = 1.0 / (counts + 1e-6)
    weights = weights / weights.sum() * len(Config.inter_names)
    return weights.float()

In [ ]:
#estrazione file e feature
class HierarchicalClassificationDataset(Dataset):
    def __init__(self, pickle_file, protein_to_idx=None):
        with open(pickle_file, "rb") as f:
            raw_data = pickle.load(f)

        if isinstance(raw_data, dict):
            if 'file_name' in raw_data and isinstance(raw_data['file_name'], list):
                data = [{'file_name': raw_data['file_name'][i], 
                         'embedding': raw_data['embedding'][i], 
                         'fitness': raw_data['fitness'][i]} 
                        for i in range(len(raw_data['file_name']))]
            else:
                data = list(raw_data.values())
        else:
            data = raw_data

        self.embeddings = []
        self.labels = [] 
        self.protein_indices = []

        if protein_to_idx is None:
            unique_proteins = set()
            for entry in data:
                prot_name = entry.get('file_name', 'unknown')[:4]
                unique_proteins.add(prot_name)
            self.protein_to_idx = {name: i for i, name in enumerate(sorted(list(unique_proteins)))}
        else:
            self.protein_to_idx = protein_to_idx

        all_fitness = []
        for entry in data:
            all_fitness.append(entry['fitness'])
            self.embeddings.append(entry['embedding'])
            prot_name = entry.get('file_name', 'unknown')[:4]
            self.protein_indices.append(self.protein_to_idx.get(prot_name, 0))

        self.labels = discretize_fitness(np.array(all_fitness))
        self.embeddings = torch.tensor(np.array(self.embeddings), dtype=torch.float32)
        self.labels = torch.tensor(self.labels, dtype=torch.long)
        self.protein_indices = torch.tensor(self.protein_indices, dtype=torch.long)

    def __len__(self): return len(self.labels)
    def __getitem__(self, idx): return self.embeddings[idx], self.labels[idx], self.protein_indices[idx]
    def get_num_proteins(self): return len(self.protein_to_idx)

In [ ]:
class HyperbolicContrastiveLoss(nn.Module):
    
    def __init__(self, manifold, temperature=0.2):
        super().__init__()
        self.manifold = manifold
        self.temperature = temperature
        self.nll_loss = nn.NLLLoss()

    def forward(self, embeddings, prototypes, labels):
        
        # 1. Calcolo delle distanze a coppie nello spazio iperbolico
        # Unsqueeze per broadcasting: (Batch, 1, Dim) vs (1, Classes, Dim)
        dists = self.manifold.dist(embeddings.unsqueeze(1), prototypes.unsqueeze(0))
        
        # 2. Convertiamo le distanze in "logits" (similarità)
        # In HycoCLIP, più sono vicini (distanza bassa), più alta è la similarità.
        # Usiamo il negativo della distanza scalato dalla temperatura.
        logits = -dists / self.temperature
        
        # 3. Log-Softmax e NLL Loss
        log_probs = F.log_softmax(logits, dim=1)
        loss = self.nll_loss(log_probs, labels)
        
        return loss

In [ ]:
def entailment_cone_loss(child_emb, parent_emb, K=4):
    parent_norm = parent_emb.norm(p=2, dim=-1).clamp_min(1e-5)
    # Clamp per evitare errori numerici in asin
    asin_arg = (K * (1 - parent_norm**2) / parent_norm).clamp(-1.0+1e-7, 1.0-1e-7) 
    cone_aperture = torch.asin(asin_arg)
    
    u_norm = F.normalize(parent_emb, p=2, dim=-1)
    v_norm = F.normalize(child_emb, p=2, dim=-1)
    
    # Clamp per evitare errori numerici in acos
    dot_prod = (u_norm * v_norm).sum(dim=-1).clamp(-1.0+1e-7, 1.0-1e-7) 
    theta = torch.acos(dot_prod)
    
    return torch.relu(theta - cone_aperture).mean()

def build_mlp(input_dim, layer_sizes, output_dim, dropout_p=0.05):
    layers = []
    in_d = input_dim
    for hidden_d in layer_sizes:
        layers.append(nn.Linear(in_d, hidden_d))
        layers.append(nn.SiLU())
        layers.append(nn.Dropout(dropout_p))
        in_d = hidden_d
    layers.append(nn.Linear(in_d, output_dim))
    return nn.Sequential(*layers)

In [ ]:
class HycoClassifier(nn.Module):
    def __init__(self, input_dim, hyp_dim, hidden_layers, num_proteins, num_classes, c=1.0):
        super().__init__()
        self.manifold = geoopt.PoincareBall(c=c)
        self.encoder = build_mlp(input_dim, hidden_layers, hyp_dim)        
        # Prototipi Geometrici per l'Entailment (rimangono quelli per le proteine/parent specifici)
        self.protein_prototypes = geoopt.ManifoldParameter(
            self.manifold.random(num_proteins, hyp_dim).mul_(0.05), manifold=self.manifold
        )
        
        # NUOVO: Prototipi per la Classificazione (Sostituisce mlp_classifier)
        # In HycoCLIP, la classificazione è la distanza dai centroidi delle classi.
        # Creiamo un parametro apprendibile per ogni classe nello spazio iperbolico.
        self.class_prototypes = geoopt.ManifoldParameter(
            self.manifold.random(num_classes, hyp_dim).mul_(0.05), manifold=self.manifold
        )
        
    def forward(self, x, protein_indices):
        x_euc = self.encoder(x)
        child_hyp = self.manifold.expmap0(x_euc) # Mappa nel disco
        
        # Recupera il prototipo specifico per la loss di Entailment
        parent_hyp = self.protein_prototypes[protein_indices]
        
        # RITORNO:
        # 1. child_hyp: l'embedding della proteina attuale
        # 2. parent_hyp: il genitore specifico per l'entailment (gerarchia)
        # 3. self.class_prototypes: TUTTI i centri delle classi per calcolare la Contrastive Loss
        return child_hyp, parent_hyp, self.class_prototypes

In [ ]:
# ==========================================
# 2. TRAINING PHASE
# ==========================================

def train_engine(model, train_loader, val_loader, model_type, class_weights):
    optimizer = torch.optim.Adam(model.parameters(), lr=Config.learning_rate)
    
    # Istanziamo la Loss
    criterion_hyp_contrastive = HyperbolicContrastiveLoss(model.manifold, temperature=0.07).to(Config.device)
    
    # SETUP SALVATAGGIO
    best_val_f1_macro = 0.0  # Ci basiamo sul Macro F1 come richiesto
    best_model_path = "best_hycoclip_model.pth" # Nome del file locale

    for epoch in range(Config.epochs):
        model.train()
        train_loss = 0
        
        for inputs, labels, prot_idx in train_loader:
            inputs = inputs.to(Config.device)
            labels = labels.to(Config.device)
            prot_idx = prot_idx.to(Config.device)
            
            optimizer.zero_grad()
            
            child_hyp, parent_hyp, class_prototypes = model(inputs, prot_idx)
    
            loss_contrastive = criterion_hyp_contrastive(child_hyp, class_prototypes, labels)
            
            loss_ent = entailment_cone_loss(child_hyp, parent_hyp, K=Config.cone_aperture_k)
            loss = loss_contrastive + (Config.lambda_entail * loss_ent)
                
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        # --- VALIDATION ---
        model.eval()
        val_preds, val_targets = [], []
        
        with torch.no_grad():
            for inputs, labels, prot_idx in val_loader:
                inputs = inputs.to(Config.device)
                labels = labels.to(Config.device)
                prot_idx = prot_idx.to(Config.device)
                
                child_hyp, _, class_prototypes = model(inputs, prot_idx)
                
                # Inferenza Contrastiva: Argmin della distanza
                dists = model.manifold.dist(child_hyp.unsqueeze(1), class_prototypes.unsqueeze(0))
                preds = torch.argmin(dists, dim=1)
                
                val_preds.extend(preds.cpu().numpy())
                val_targets.extend(labels.cpu().numpy())
        
        # Calcolo Metriche
        val_f1_macro = f1_score(val_targets, val_preds, average='macro')
        val_f1_weighted = f1_score(val_targets, val_preds, average='weighted')
        val_rho, _ = spearmanr(val_targets, val_preds)
        
        # --- SALVATAGGIO MIGLIOR MODELLO (Basato su F1 MACRO) ---
        if val_f1_macro > best_val_f1_macro:
            best_val_f1_macro = val_f1_macro
            # Salva su disco
            torch.save(model.state_dict(), best_model_path)
            # (Opzionale) Feedback visivo immediato quando si trova un nuovo best
            # print(f"  -> New Best F1 Macro: {val_f1_macro:.4f} (Saved to {best_model_path})")
            
        if (epoch+1) % 5 == 0:
            print(f"Ep {epoch+1} | Loss: {train_loss/len(train_loader):.4f} | Val F1 Macro: {val_f1_macro:.4f} (Best: {best_val_f1_macro:.4f}) | Rho: {val_rho:.4f}")
            
    # Alla fine, ricarichiamo il modello migliore salvato su disco
    print(f"\nTraining Complete. Loading best model from '{best_model_path}'...")
    model.load_state_dict(torch.load(best_model_path))
    
    return model, best_val_f1_macro

def main():
    full_dataset = HierarchicalClassificationDataset("finalized_pooled_embeddings.pkl")
    sample_emb, _, _ = full_dataset[0]
    Config.input_dim = sample_emb.shape[0]

    train_size = int(0.8 * len(full_dataset))
    train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, len(full_dataset)-train_size])
    train_loader = DataLoader(train_dataset, batch_size=Config.batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=Config.batch_size, shuffle=False)
    class_weights = compute_class_weights(train_loader, Config.device)

    # Inizializzazione Modello
    hyp_model = HycoClassifier(Config.input_dim, Config.hyp_dim, Config.classifier_layers, 
                            full_dataset.get_num_proteins(), Config.output_dim).to(Config.device)

    # Avvio Training
    hyp_model, best_f1_hyp = train_engine(hyp_model, train_loader, val_loader, "hyperbolic", class_weights)

    return hyp_model, val_loader

In [ ]:
if __name__ == "__main__":
    hyp_model, val_loader = main()

In [ ]:
# ==================================================================================
# 3. VISUALIZATION & METRICS
# ==================================================================================

def visualize_confusion_matrix(model, loader, device, model_type):
    model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for inputs, labels, prot_idx in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            prot_idx = prot_idx.to(device)
            child_hyp, _, class_prototypes = model(inputs, prot_idx)
            dists = model.manifold.dist(child_hyp.unsqueeze(1), class_prototypes.unsqueeze(0)) 
            preds = torch.argmin(dists, dim=1)  
            all_preds.extend(preds.cpu().numpy())
            all_true.extend(labels.cpu().numpy())
            
    cm = confusion_matrix(all_true, all_preds, labels=range(Config.output_dim))
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=Config.inter_names, yticklabels=Config.inter_names)
    plt.title(f'Confusion Matrix ({model_type.capitalize()} - Contrastive Inf.)')
    plt.ylabel('True')
    plt.xlabel('Predicted')
    plt.show()

def visualize_poincare_embeddings(model, loader, device):
    model.eval()
    embeddings_list = []
    labels_list = []

    with torch.no_grad():
        for inputs, labels, prot_idx in loader:
            inputs = inputs.to(device)
            prot_idx = prot_idx.to(device)
            
            child_hyp, _, _ = model(inputs, prot_idx)
            
            tangent_vecs = model.manifold.logmap0(child_hyp)
            
            embeddings_list.append(tangent_vecs.cpu().numpy())
            labels_list.append(labels.numpy())
            
    X = np.concatenate(embeddings_list, axis=0)
    y = np.concatenate(labels_list, axis=0)
    
    # applying the PCA to reduce components
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X)
    
    # Plotting
    plt.figure(figsize=(10, 8))
    # Disegniamo il cerchio unitario (bordo del disco di Poincaré)
    circle = plt.Circle((0, 0), 1.0, color='black', fill=False, linestyle='--', alpha=0.3)
    plt.gca().add_patch(circle)
    
    # Normalizzazione per visualizzazione nel disco
    max_val = np.max(np.abs(X_pca))
    X_pca_norm = X_pca / (max_val * 1.1) 
    
    scatter = plt.scatter(X_pca_norm[:, 0], X_pca_norm[:, 1], c=y, 
                          cmap=ListedColormap(Config.label_col), s=20, alpha=0.7)
    
    plt.legend(handles=[mpatches.Patch(color=Config.label_col[i], label=Config.inter_names[i]) 
                        for i in range(len(Config.inter_names))], loc='upper right')
    plt.title("Hyperbolic Space Projection (Tangent PCA -> 2D Disk)")
    plt.axis('equal')
    plt.axis('off') # Cleaner look
    plt.show()

def get_metrics(model, loader, device, model_type):
    model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for inputs, labels, prot_idx in loader:
            inputs, labels, prot_idx = inputs.to(device), labels.to(device), prot_idx.to(device)
            
            if model_type == "hyperbolic":
                # Logica Contrastiva: Distanza Minima
                child_hyp, _, class_prototypes = model(inputs, prot_idx)
                dists = model.manifold.dist(child_hyp.unsqueeze(1), class_prototypes.unsqueeze(0))
                preds = torch.argmin(dists, dim=1)
            else:
                logits = model(inputs)
                preds = torch.argmax(logits, dim=1)
                
            all_preds.extend(preds.cpu().numpy())
            all_true.extend(labels.cpu().numpy())
            
    return {
        'F1 Weighted': f1_score(all_true, all_preds, average='weighted'),
        'F1 Macro': f1_score(all_true, all_preds, average='macro'),
        'Spearman Rho': spearmanr(all_true, all_preds)[0]
    }

# Confusion matrix
print("Generating Confusion Matrix...")
visualize_confusion_matrix(hyp_model, val_loader, Config.device, "hyperbolic")

# Metrics table
print("Calculating Metrics...")
m_hyp = get_metrics(hyp_model, val_loader, Config.device, "hyperbolic")

print("\n" + "="*45)
print(f"{'METRIC':<20} | {'HYPERBOLIC':<10}")
print("-" * 45)
for key in m_hyp.keys():
    print(f"{key:<20} | {m_hyp[key]:.4f}")
print("="*45)

# Projection on the 2D circle
print("Generating Poincaré Projection...")
visualize_poincare_embeddings(hyp_model, val_loader, Config.device)